# 5주차 직접 해보기: 사전 학습 언어 모델

사이트의 5주차 회독과 정리 슬라이드를 본 다음 풀어요. 위에서부터 차례로 `Shift+Enter` 로 실행해요.

- 문제마다 **생각 순서**가 먼저 나와요. 코드를 치기 전에 그 순서를 말로 한 번 읊어요.
- `# TODO` 가 있는 칸의 `None` 을 알맞은 코드로 바꿔요.
- 바로 아래 **확인 셀**을 실행하면 맞았는지 알려 줘요. 틀리면 빨간 AssertionError 와 힌트가 나와요.
- 막히면 맨 아래 **정답 코드**를 봐요.

GPU 도, 인터넷 다운로드도, 모델 내려받기도 없어요. 실습 노트북 Lab 4 와 달리 여기서는 숫자만 손으로 따라가요.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
torch.manual_seed(0)
def ok(cond, msg):
    assert cond, msg
    print('정답! ' + msg)
print('준비 끝')

## 1. 15 퍼센트 마스킹과 80/10/10 세기

강의 N5 p.19-20. **마스크 언어 모델(Masked Language Modelling (MLM))** 은 토큰의 15 퍼센트를 골라서 가리고, 그 자리를 맞히게 해요.

고른 15 퍼센트가 전부 **마스크 토큰([MASK])** 이 되는 것은 아니에요. 그중 80 퍼센트만 [MASK] 가 되고, 10 퍼센트는 아무 토큰으로 바뀌고, 10 퍼센트는 그대로 둬요. 세 경우 모두 원래 단어를 맞혀야 해요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 먼저 몇 개를 고르는지 세요. 토큰 200개의 15 퍼센트예요
2. 그 30개를 다시 80 대 10 대 10 으로 나눠요
3. 80 퍼센트는 30 의 0.8 이라서 24개예요
4. 10 퍼센트는 3개씩이고, 24 더하기 3 더하기 3 이 30 이어야 해요
5. 손실을 세는 자리는 [MASK] 가 된 24개가 아니라 고른 30개 전부예요

> **나중에 여기서 만나요** N5 p.20 이 80/10/10 그림이고, 미세조정 때는 [MASK] 가 아예 없어서 모델이 [MASK] 를 보는 데 기대면 안 된다는 것이 이유예요. 실습 N5L p.13 에서 실제로 빈칸을 채워 봐요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
고른 수 = 전체 토큰 수 곱하기 0.15
마스크로 바꾸는 수 = 고른 수 곱하기 0.8
아무 토큰으로 바꾸는 수 = 고른 수 곱하기 0.1
그대로 두는 수 = 고른 수 빼기 앞의 둘
손실을 세는 자리 수 = 고른 수
```

In [ ]:
n_tokens = 200

chosen = None      # TODO: 15 퍼센트를 고르기 (int() 로 버림)
to_mask = None     # TODO: 고른 것의 80 퍼센트
to_random = None   # TODO: 고른 것의 10 퍼센트
to_keep = None     # TODO: 나머지 (고른 수에서 둘을 빼기)
loss_positions = None   # TODO: 손실을 세는 자리 수

print(chosen, to_mask, to_random, to_keep, loss_positions)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(chosen == 30, '200 의 15 퍼센트는 30개예요')
ok(to_mask == 24, '30 의 80 퍼센트는 24개가 [MASK] 가 돼요')
ok(to_random == 3 and to_keep == 3, '10 퍼센트씩 3개는 아무 토큰으로, 3개는 그대로 둬요')
ok(to_mask + to_random + to_keep == chosen, '셋을 더하면 고른 30개가 다시 나와요')
ok(loss_positions == 30, '손실은 [MASK] 24자리가 아니라 고른 30자리 전부에서 세요')

## 2. 마스크 자리에서만 손실 세기

강의 N5 p.19. **손실 함수(Loss Function)** 는 가린 자리에서만 계산해요. 가리지 않은 자리는 답이 그대로 보이니까 점수를 매길 이유가 없어요.

한 자리의 손실은 2주차, 3주차에서 본 그대로예요. 어휘 전체에 **소프트맥스(Softmax)** 를 씌워 확률을 만들고, 정답 토큰에 준 확률의 음의 로그가 **교차 엔트로피(Cross-Entropy)** 예요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 어휘가 5개라서 한 자리마다 점수가 5개 나와요
2. 줄마다 소프트맥스를 해요. 줄 합이 1 이 되어야 해요
3. 첫 줄의 정답은 0번 칸, 둘째 줄의 정답은 1번 칸이에요
4. 정답 칸의 확률을 꺼내고 음의 로그를 씌워요
5. 가린 자리가 두 개니까 두 값을 평균 내요. 나머지 세 자리는 아예 안 세요

> **나중에 여기서 만나요** N2 p.33 의 소프트맥스와 N3 p.41 의 교차 엔트로피가 그대로 쓰여요. 실습 N5L p.13 의 빈칸 채우기 출력에 나오는 서울 0.64 같은 숫자가 바로 이 확률이에요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
줄마다 최댓값을 빼고 exp 를 씌운 뒤 줄 합으로 나눈다
정답 칸의 확률만 꺼낸다
음의 로그를 씌운다
가린 자리 수로 평균을 낸다
```

In [ ]:
scores = np.array([[2.0, 1.0, 0.0, 0.0, 1.0],
                   [0.0, 3.0, 1.0, 0.0, 0.0]])
gold = np.array([0, 1])

def row_softmax(M):
    M = M - M.max(axis=1, keepdims=True)
    e = np.exp(M)
    return e / e.sum(axis=1, keepdims=True)

P = None          # TODO: 줄마다 소프트맥스
p_gold = None     # TODO: 정답 칸의 확률만 꺼내기 (P[np.arange(2), gold])
loss = None       # TODO: 음의 로그를 씌워 평균 내기

print(P.round(4))
print(p_gold.round(4), round(float(loss), 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(np.allclose(P.sum(axis=1), 1.0), '줄마다 합이 정확히 1 이에요')
ok(np.allclose(p_gold, [0.4984, 0.7784], atol=1e-4), '정답 확률은 0.4984 와 0.7784 예요')
ok(abs(float(loss) - 0.4734) < 1e-4, '두 자리 손실의 평균은 약 0.4734 예요')
ok(abs(float(loss) - float(-np.log(P[0, 0]) - np.log(P[1, 1])) / 2) < 1e-12, '가린 두 자리만 더해서 2 로 나눈 값이에요')

## 3. 양쪽을 다 보는 BERT, 앞만 보는 GPT

강의 N5 p.17-18, 4주차 N4 p.43. 4주차에서 **인코더(Encoder)** 는 양쪽을 다 보고 **디코더(Decoder)** 는 앞만 본다고 배웠죠. 그 차이가 5주차에서 **버트(BERT)** 와 **지피티(GPT)** 로 갈려요.

디코더는 **인과 마스킹(Causal Masking)** 때문에 오른쪽 위가 막혀 있어요. 인코더는 막힌 데가 없어서 다음 단어 맞히기가 아예 문제가 되지 않아요. 답이 이미 입력에 보이니까요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 토큰 4개짜리 문장 하나로 해 봐요. 표는 4 곱하기 4 예요
2. BERT 는 모든 칸이 1 이에요. np.ones 로 만들면 끝이에요
3. GPT 는 아래쪽 삼각형만 1 이에요. np.tril 이 그 표를 만들어요
4. 볼 수 있는 칸을 세면 BERT 는 16, GPT 는 1 더하기 2 더하기 3 더하기 4 로 10 이에요
5. GPT 의 줄마다 개수를 찍어 보면 1, 2, 3, 4 로 늘어나요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
bert = 4 곱하기 4 짜리 전부 1 인 표
gpt = 같은 크기에서 아래쪽 삼각형만 1 인 표
각각 다 더해서 볼 수 있는 칸 수를 센다
gpt 는 줄마다 합을 따로 찍어 본다
```

> **나중에 여기서 만나요** N4 p.43 의 인과 마스킹이 GPT 표를 만들고, N5 p.18 이 그래서 인코더로는 다음 단어 맞히기를 못 한다고 말해요. N5 p.30 의 디코더 목적 함수는 3주차 N3 p.46-47 의 RNN 언어 모델과 같은 일이에요.

In [ ]:
n = 4

bert_mask = None   # TODO: 4 x 4 전부 1 (np.ones)
gpt_mask = None    # TODO: 아래쪽 삼각형만 1 (np.tril 에 np.ones 를 넣기)

bert_seen = None   # TODO: 볼 수 있는 칸 수 (int(...sum()))
gpt_seen = None    # TODO: 볼 수 있는 칸 수

print(gpt_mask)
print(bert_seen, gpt_seen, gpt_mask.sum(axis=1))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(bert_seen == 16, 'BERT 는 16칸을 다 봐요. 그래서 다음 단어가 이미 보여요')
ok(gpt_seen == 10, 'GPT 는 10칸만 봐요. 1 더하기 2 더하기 3 더하기 4 예요')
ok(np.allclose(gpt_mask.sum(axis=1), [1, 2, 3, 4]), '줄마다 1, 2, 3, 4 로 늘어나요')
ok(np.allclose(np.triu(gpt_mask, 1), 0.0), '오른쪽 위, 곧 미래는 전부 0 이에요')

## 4. BERT 를 숫자로 읽기

강의 N5 p.22. **버트(BERT)** 는 두 가지 크기로 나왔어요.

| | 층 | 은닉 | 헤드 | 파라미터 |
|---|---|---|---|---|
| BERT-base | 12 | 768 | 12 | 110M |
| BERT-large | 24 | 1024 | 16 | 340M |

4주차 N4 p.47 에서 **멀티 헤드 어텐션(Multi-Head Attention (MHA))** 은 전체 차원을 헤드 수로 나눠 쓴다고 배웠어요. 그 규칙을 여기 숫자에 그대로 대 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 헤드 하나가 쓰는 칸 수는 은닉을 헤드 수로 나눈 값이에요
2. base 는 768 나누기 12, large 는 1024 나누기 16 이에요
3. 두 값이 같은지 보세요. 둘 다 64 가 나와요
4. large 가 base 보다 파라미터가 몇 배인지도 나눠 보면 돼요
5. 340 나누기 110 은 약 3.09 배예요. 층은 2배인데 파라미터는 3배가 넘어요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
base 헤드 차원 = 768 나누기 12
large 헤드 차원 = 1024 나누기 16
파라미터 배수 = 340 나누기 110
세 값을 찍어서 비교한다
```

> **나중에 여기서 만나요** N4 p.47 의 d 나누기 h 규칙이 그대로예요. 실습 N5L p.10 에서 쓰는 `klue/bert-base` 가 바로 이 표의 왼쪽 줄, 곧 층 12, 은닉 768, 파라미터 110M 짜리 한국어 BERT 예요.

In [ ]:
base = {"layers": 12, "hidden": 768, "heads": 12, "params_m": 110}
large = {"layers": 24, "hidden": 1024, "heads": 16, "params_m": 340}

base_head_dim = None    # TODO: 은닉을 헤드 수로 나누기 (// 사용)
large_head_dim = None   # TODO: 같은 방식으로
param_ratio = None      # TODO: large 파라미터가 base 의 몇 배인지

print(base_head_dim, large_head_dim, round(param_ratio, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(base_head_dim == 64, '768 나누기 12 는 64. 헤드 하나가 64칸을 봐요')
ok(large_head_dim == 64, '1024 나누기 16 도 64 예요. 크기를 키워도 헤드 한 개의 칸 수는 그대로예요')
ok(abs(param_ratio - 3.0909) < 1e-3, '340 나누기 110 은 약 3.09 배예요')
ok(base["layers"] * 2 == large["layers"], '층은 정확히 2배인데 파라미터는 3배가 넘어요')

## 5. 몸통은 그대로, 과제 헤드만 갈아 끼우기

강의 N5 p.23. 같은 **몸통(Body)** 위에 **과제 헤드(Task Head)** 만 바꿔 끼우면 거의 모든 과제가 돼요. 공구 자루 하나에 날만 갈아 끼우는 것과 같아요.

**인코더(Encoder)** 가 내놓는 것은 토큰마다 768칸짜리 줄이에요. 문장 하나를 분류할 때는 **분류 토큰([CLS])** 자리만 쓰고, 토큰마다 꼬리표를 붙일 때는 모든 자리를 써요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 인코더 출력 모양부터 적어요. (문장 4개, 토큰 16개, 768칸) 이에요
2. 문장 분류는 [CLS] 자리 한 줄만 꺼내요. x[:, 0] 이 그 자리예요
3. 토큰 태깅은 모든 자리에 그대로 씌워요. 모양이 (4, 16, 태그 수) 가 돼요
4. 헤드 파라미터는 Linear(768, k) 라서 768 곱하기 k 에 편향 k 를 더한 값이에요
5. 2 클래스면 1538개. 몸통 1억 1천만 개에 견주면 거의 없는 수준이에요

> **나중에 여기서 만나요** N5 p.23 의 네 가지 과제 모양(문장 하나, 문장 쌍, 토큰 태깅, 스팬 추출)이 이 계산이에요. 실습 N5L p.14 에서 Hugging Face 가 마스크 언어 모델 헤드를 버리고 2 클래스 헤드를 새로 붙이는 장면이 바로 이것이고, N5L p.15 의 학습 전 정확도 57.0 퍼센트가 그 새 헤드가 아직 무작위라는 증거예요.

### 의사코드 먼저

이번에는 순서를 스스로 세워 봐요. 위 **생각 순서**를 보고 의사코드를 서너 줄로 적은 다음에 코드를 쳐요. 모범 의사코드는 맨 아래 정답 모음에 있어요.

In [ ]:
enc_out = torch.zeros(4, 16, 768)    # (문장 4개, 토큰 16개, 768칸)

cls_head = nn.Linear(768, 2)         # 문장 하나 분류
tag_head = nn.Linear(768, 9)         # 토큰마다 꼬리표 9종

cls_logits = None    # TODO: [CLS] 자리(enc_out[:, 0])에만 cls_head 를 씌우기
tag_logits = None    # TODO: 모든 자리에 tag_head 를 씌우기

cls_params = None    # TODO: 768 곱하기 2 더하기 2
body_params = 110_000_000
head_share = None    # TODO: cls_params 가 body_params 의 몇 퍼센트인지

print(tuple(cls_logits.shape), tuple(tag_logits.shape))
print(cls_params, round(head_share, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(tuple(cls_logits.shape) == (4, 2), '문장 분류는 문장마다 답 하나라서 (4, 2)')
ok(tuple(tag_logits.shape) == (4, 16, 9), '토큰 태깅은 토큰마다 답이라서 (4, 16, 9)')
ok(cls_params == 1538, '768 곱하기 2 에 편향 2 를 더해 1538개')
ok(cls_params == sum(p.numel() for p in cls_head.parameters()), 'PyTorch 가 센 수와 같아요')
ok(head_share < 0.01, '몸통의 0.01 퍼센트도 안 돼요. 새로 배우는 것은 이것뿐이에요')

## 6. 미세조정 레시피를 걸음 수로 바꾸기

강의 N5 p.44. **미세조정(Finetuning)** 은 네 숫자면 끝이에요. **학습률(Learning Rate)** 2e-5 에서 5e-5, **에폭(Epoch)** 2 에서 4, **배치 크기(Batch Size)** 16 또는 32, 그리고 처음 약 10 퍼센트 걸음은 **워밍업(Warmup)** 이에요.

실습 N5L p.20 이 정확히 이 숫자로 돌아가요. 레이블 15,000개, 배치 32, 에폭 2 일 때 몇 걸음을 걷는지 세어 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 한 에폭은 데이터를 한 번 다 보는 것이에요
2. 한 에폭의 걸음 수는 15000 나누기 32 인데, 남는 것도 한 걸음이라 올림이에요
3. math.ceil(15000 / 32) 은 469 예요. 468.75 를 올린 값이에요
4. 에폭이 2 이니까 전체 걸음은 469 곱하기 2 로 938 이에요
5. 워밍업은 전체의 10 퍼센트를 버림해서 93 걸음이에요
6. 미세조정 학습률이 사전 학습보다 100배 작다고 했으니, 거꾸로 100을 곱하면 사전 학습 쪽 보폭이에요

> **나중에 여기서 만나요** 실습 N5L p.20 의 `warmup_steps=max(1, int(0.1 * total_steps))` 가 그대로 이 계산이에요. N5 p.44 는 가장 흔한 실수가 사전 학습 학습률을 그대로 쓰는 것이라고 경고해요. 그러면 모델이 알던 것이 지워져요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
한 에폭 걸음 수 = 데이터 수 나누기 배치 크기, 올림
전체 걸음 수 = 한 에폭 걸음 수 곱하기 에폭 수
워밍업 걸음 수 = 전체 걸음 수 곱하기 0.1, 버림
사전 학습 학습률 = 미세조정 학습률 곱하기 100
```

In [ ]:
n_train, batch, epochs = 15000, 32, 2
lr_finetune = 2e-5

steps_per_epoch = None   # TODO: 올림 (math.ceil)
total_steps = None       # TODO: 에폭 수만큼 곱하기
warmup_steps = None      # TODO: 전체의 10 퍼센트, 버림 (int)
lr_pretrain = None       # TODO: 미세조정보다 100배 큰 값

print(steps_per_epoch, total_steps, warmup_steps, lr_pretrain)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(steps_per_epoch == 469, '15000 나누기 32 는 468.75, 올려서 469 걸음이에요')
ok(total_steps == 938, '에폭 2 니까 938 걸음이에요')
ok(warmup_steps == 93, '938 의 10 퍼센트를 버림하면 93 걸음이에요')
ok(abs(lr_pretrain - 2e-3) < 1e-12, '미세조정 2e-5 의 100배는 2e-3 이에요. 이 보폭으로 미세조정하면 안 돼요')

## 7. 1 퍼센트만 움직이기

강의 N5 p.46. **파라미터 효율적 미세조정(Parameter-efficient Finetuning (PEFT))** 은 사전 학습된 **몸통(Body)** 을 **동결(Freezing)** 하고 작은 **어댑터(Adapter)** 만 학습해요. 움직이는 가중치가 전체의 약 1 퍼센트예요.

왜 좋은지는 과제가 여러 개일 때 확 드러나요. 전체 미세조정은 과제마다 모델을 통째로 한 벌씩 저장해야 하지만, PEFT 는 몸통 한 벌에 작은 조각만 여러 개 붙여 두면 돼요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 몸통은 1억 1천만 개예요. 파이썬으로는 110_000_000 이라고 써도 돼요
2. 1 퍼센트는 0.01 을 곱한 값이라서 110만 개예요
3. 전체 미세조정은 몸통 전부가 움직이니 110만 개의 100배를 움직여요
4. 과제 10개를 전부 전체 미세조정하면 110M 짜리를 10벌 저장해요
5. PEFT 는 몸통 1벌에 110만 짜리 조각 10개만 더하면 돼요
6. 두 저장 용량을 나눠 보면 9배가 넘게 차이가 나요

> **나중에 여기서 만나요** N5 p.46 이 1 퍼센트라는 숫자가 나오는 쪽이고, LoRA 와 어댑터가 그 방법이에요. N5 p.48 의 '한 번 비싸게 배우고 계속 다시 쓴다' 가 이 계산으로 설명돼요. 이 과목에서 직접 PEFT 를 돌리지는 않아요.

### 의사코드 먼저

이번에는 순서를 스스로 세워 봐요. 위 **생각 순서**를 보고 의사코드를 서너 줄로 적은 다음에 코드를 쳐요. 모범 의사코드는 맨 아래 정답 모음에 있어요.

In [ ]:
body = 110_000_000
tasks = 10

trainable = None      # TODO: 몸통의 1 퍼센트 (int 로)
fewer_times = None    # TODO: 몸통이 trainable 의 몇 배인지 (// 사용)
full_store = None     # TODO: 전체 미세조정으로 과제 10개를 저장할 때
peft_store = None     # TODO: PEFT 로 과제 10개를 저장할 때 (몸통 1벌 + 조각 10개)

print(trainable, fewer_times)
print(full_store, peft_store, round(full_store / peft_store, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(trainable == 1_100_000, '1억 1천만의 1 퍼센트는 110만 개예요')
ok(fewer_times == 100, '몸통은 그 100배예요')
ok(full_store == 1_100_000_000, '과제 10개면 11억 개를 저장해요')
ok(peft_store == 121_000_000, 'PEFT 는 1억 2천 1백만 개면 끝이에요')
ok(abs(full_store / peft_store - 9.0909) < 1e-3, '약 9.09배 차이예요')

## 8. 혼동 행렬에서 네 가지 숫자 구하기

실습 N5L p.24. **정확도(Accuracy)** 하나로는 언제 틀리는지 알 수 없어요. **혼동 행렬(Confusion Matrix)** 로 쪼개 봐야 해요.

먼저 노트북에 저장된 보고서를 거꾸로 풀어 봐요. 부정 리뷰 2516건의 **재현율(Recall)** 이 0.874, 긍정 리뷰 2484건의 재현율도 0.874 였어요. 재현율은 '실제로 그 반인 것 중 몇 개를 맞혔나' 예요.

그다음에는 연습용 작은 표로 네 숫자를 모두 구해 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 재현율 곱하기 실제 개수가 맞힌 개수예요
2. 부정은 0.874 곱하기 2516 이라서 약 2199 건이에요
3. 긍정은 0.874 곱하기 2484 라서 약 2171 건이에요
4. 둘을 더해 전체 5000 으로 나누면 정확도 0.874 가 그대로 나와요
5. 작은 표에서는 오른쪽 아래가 맞힌 긍정, 오른쪽 위가 긍정이라고 잘못 부른 것이에요
6. 정밀도는 긍정이라고 부른 것 중 맞은 비율, 재현율은 실제 긍정 중 맞은 비율이에요
7. F1 은 두 값의 조화 평균이라 2pr 나누기 (p 더하기 r) 예요

> **나중에 여기서 만나요** 실습 N5L p.24 의 보고서가 이 숫자이고, N5L p.25 는 확신까지 하면서 틀린 12건을 꺼내 읽어요. 노트북 본문은 정확도를 87.5 퍼센트라고 적었지만 저장된 출력은 0.8740 이에요. 출력 숫자를 따라요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
맞힌 부정 = 0.874 곱하기 2516, 반올림
맞힌 긍정 = 0.874 곱하기 2484, 반올림
정확도 = (둘의 합) 나누기 5000
작은 표에서 정확도 = 대각선 합 나누기 전체
정밀도 = 오른쪽 아래 나누기 오른쪽 열 합
재현율 = 오른쪽 아래 나누기 아랫줄 합
F1 = 2 곱하기 정밀도 곱하기 재현율 나누기 (정밀도 더하기 재현율)
```

In [ ]:
rec_neg, n_neg = 0.874, 2516
rec_pos, n_pos = 0.874, 2484

correct_neg = None   # TODO: 재현율 곱하기 개수, 반올림 (round)
correct_pos = None   # TODO: 같은 방식
accuracy = None      # TODO: 둘을 더해 5000 으로 나누기

cm = np.array([[45, 5],
               [10, 40]])   # 줄: 실제 부정, 실제 긍정 / 칸: 부정이라 부름, 긍정이라 부름

acc2 = None      # TODO: 대각선 합 나누기 전체 합
precision = None # TODO: cm[1, 1] 나누기 긍정이라 부른 것 전체 (cm[:, 1].sum())
recall = None    # TODO: cm[1, 1] 나누기 실제 긍정 전체 (cm[1].sum())
f1 = None        # TODO: 2 곱하기 precision 곱하기 recall 나누기 (precision 더하기 recall)

print(correct_neg, correct_pos, accuracy)
print(acc2, round(precision, 4), recall, round(f1, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(correct_neg == 2199 and correct_pos == 2171, '부정 2199건, 긍정 2171건을 맞혔어요')
ok(abs(accuracy - 0.874) < 1e-12, '4370 나누기 5000 은 정확히 0.874. 노트북 출력과 같아요')
ok(abs(acc2 - 0.85) < 1e-12, '작은 표의 정확도는 85 나누기 100 으로 0.85 예요')
ok(abs(precision - 40 / 45) < 1e-12, '긍정이라 부른 45건 중 40건이 맞아서 약 0.8889')
ok(abs(recall - 0.8) < 1e-12, '실제 긍정 50건 중 40건을 찾아서 0.8')
ok(abs(f1 - 16 / 19) < 1e-12, 'F1 은 약 0.8421. 정밀도와 재현율 사이에 있어요')

## 9. 레이블을 늘리면 얼마나 좋아질까

강의 N5 p.45, 실습 N5L p.30. 사전 학습은 레이블을 사 줘요. 몇 백 개만 있어도 꽤 멀리 가요.

실습에서 같은 모델을 레이블 500개, 2,000개, 15,000개로 각각 미세조정했더니 정확도가 0.8050, 0.8418, 0.8740 이 나왔어요. **수확 체감(Diminishing Returns)** 이 어떤 모습인지 숫자로 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 상승폭은 뒤 정확도에서 앞 정확도를 뺀 값이에요. 퍼센트 포인트로 보려면 100을 곱해요
2. 500 에서 2000 은 3.68 포인트, 2000 에서 15000 은 3.22 포인트 올라요
3. 그런데 데이터는 각각 4배, 7.5배가 들었어요
4. 상승폭은 비슷한데 값이 점점 비싸지는 게 수확 체감이에요
5. 500개만으로 15000개 성능의 몇 퍼센트인지도 나눠 봐요. 92 퍼센트가 넘어요
6. 데이터는 3.3 퍼센트만 썼는데 성능은 92 퍼센트예요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
상승폭 = (뒤 정확도 빼기 앞 정확도) 곱하기 100
데이터 배수 = 뒤 레이블 수 나누기 앞 레이블 수
성능 비율 = 500 정확도 나누기 15000 정확도, 곱하기 100
데이터 비율 = 500 나누기 15000, 곱하기 100
```

> **나중에 여기서 만나요** N5 p.45 의 그림은 schematic, 곧 눈금이 실제 측정값이 아니라고 슬라이드에 적혀 있어요. 실습 N5L p.30, p.31 이 진짜 측정값이에요. 노트북 본문 표는 80.5, 84.1, 87.5 로 반올림해 적었는데, 저장된 출력은 0.8050, 0.8418, 0.8740 이에요.

In [ ]:
acc = {500: 0.8050, 2000: 0.8418, 15000: 0.8740}

gain1 = None      # TODO: 500 에서 2000 으로 갈 때 오른 퍼센트 포인트
gain2 = None      # TODO: 2000 에서 15000 으로 갈 때
cost1 = None      # TODO: 데이터가 몇 배 들었는지 (2000 나누기 500)
cost2 = None      # TODO: 15000 나누기 2000

share_perf = None # TODO: 500 정확도가 15000 정확도의 몇 퍼센트인지
share_data = None # TODO: 500 이 15000 의 몇 퍼센트인지

print(round(gain1, 2), round(gain2, 2), cost1, cost2)
print(round(share_perf, 1), round(share_data, 1))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(gain1 - 3.68) < 1e-6, '500 에서 2000 은 3.68 포인트 올라요')
ok(abs(gain2 - 3.22) < 1e-6, '2000 에서 15000 은 3.22 포인트 올라요')
ok(cost1 == 4.0 and cost2 == 7.5, '같은 크기 상승에 데이터가 4배, 그다음 7.5배 들었어요')
ok(abs(share_perf - 92.1) < 0.05, '500개만으로 15000개 성능의 약 92.1 퍼센트예요')
ok(abs(share_data - 3.3) < 0.05, '데이터는 3.3 퍼센트만 썼어요. 첫 몇 백 개가 가장 값져요')

## 10. T5 의 스팬 손상 만들어 보기

강의 N5 p.37. **티파이브(T5)** 는 토큰 하나가 아니라 **스팬(Span)**, 곧 이어진 여러 토큰을 통째로 지워요. 지운 자리마다 서로 다른 **센티널 토큰(Sentinel Token)** 을 넣고, **디코더(Decoder)** 가 빠진 조각만 써 내게 해요.

슬라이드 예시를 그대로 코드로 만들어 봐요. `Thank you for inviting me to your party last week` 에서 `for inviting` 과 `last` 를 지워요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 토큰 목록을 왼쪽부터 훑어요
2. 지금 자리가 스팬의 시작이면 센티널을 넣고, 그 스팬 끝까지 건너뛰어요
3. 스팬이 아니면 토큰을 그대로 인코더 입력에 넣어요
4. 디코더 쪽은 센티널을 쓰고, 그 뒤에 지운 토큰들을 이어 붙여요
5. 맨 끝에는 다음 센티널 하나를 더 붙여서 끝을 알려요
6. 센티널 이름은 슬라이드처럼 X, Y, Z 를 꺾쇠에 넣어 써요

> **나중에 여기서 만나요** N5 p.37 이 이 그림이고, N5 p.36 의 인코더 디코더는 4주차 N4 p.52-53 에서 이미 만든 구조예요. N5 p.17 의 표에서 인코더 디코더 칸의 목적 함수가 바로 이 스팬 손상이에요. BERT 의 마스킹(N5 p.19)과 견줘 보면 어느 쪽이 더 어려운 문제인지 보여요.

### 의사코드 먼저

이번에는 순서를 스스로 세워 봐요. 위 **생각 순서**를 보고 의사코드를 서너 줄로 적은 다음에 코드를 쳐요. 모범 의사코드는 맨 아래 정답 모음에 있어요.

In [ ]:
tokens = ["Thank", "you", "for", "inviting", "me", "to", "your", "party", "last", "week"]
spans = [(2, 4), (8, 9)]          # 반열린 구간. (2, 4) 는 for, inviting
sentinels = ["<X>", "<Y>", "<Z>"]

enc, dec, i, k = [], [], 0, 0
while i < len(tokens):
    if k < len(spans) and i == spans[k][0]:
        enc.append(None)                      # TODO: 이 자리에 넣을 센티널
        dec.append(sentinels[k])
        dec += None                           # TODO: 지운 토큰들 (tokens 의 스팬 구간)
        i = spans[k][1]
        k += 1
    else:
        enc.append(tokens[i])
        i += 1
dec.append(sentinels[k])

print("encoder input :", " ".join(enc))
print("decoder target:", " ".join(dec))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(enc == ['Thank', 'you', '<X>', 'me', 'to', 'your', 'party', '<Y>', 'week'], '인코더 입력이 슬라이드 그림과 같아요')
ok(dec == ['<X>', 'for', 'inviting', '<Y>', 'last', '<Z>'], '디코더는 빠진 조각만 써요')
ok(len(enc) == 9 and len(dec) == 6, '원문 10개가 인코더 9개, 디코더 6개가 돼요')
ok(all(t not in dec for t in ('Thank', 'you', 'me', 'week')), '지우지 않은 토큰은 디코더가 다시 쓰지 않아요')

## 정답 코드

먼저 스스로 풀어 보고, 막혔을 때만 봐요.

**1. 15 퍼센트 마스킹과 80/10/10 세기**

의사코드

```
고른 수 = 전체 토큰 수 곱하기 0.15
마스크로 바꾸는 수 = 고른 수 곱하기 0.8
아무 토큰으로 바꾸는 수 = 고른 수 곱하기 0.1
그대로 두는 수 = 고른 수 빼기 앞의 둘
손실을 세는 자리 수 = 고른 수
```

```python
n_tokens = 200

chosen = int(0.15 * n_tokens)
to_mask = int(0.8 * chosen)
to_random = int(0.1 * chosen)
to_keep = chosen - to_mask - to_random
loss_positions = chosen

print(chosen, to_mask, to_random, to_keep, loss_positions)
```

**2. 마스크 자리에서만 손실 세기**

의사코드

```
줄마다 최댓값을 빼고 exp 를 씌운 뒤 줄 합으로 나눈다
정답 칸의 확률만 꺼낸다
음의 로그를 씌운다
가린 자리 수로 평균을 낸다
```

```python
scores = np.array([[2.0, 1.0, 0.0, 0.0, 1.0],
                   [0.0, 3.0, 1.0, 0.0, 0.0]])
gold = np.array([0, 1])

def row_softmax(M):
    M = M - M.max(axis=1, keepdims=True)
    e = np.exp(M)
    return e / e.sum(axis=1, keepdims=True)

P = row_softmax(scores)
p_gold = P[np.arange(2), gold]
loss = -np.log(p_gold).mean()

print(P.round(4))
print(p_gold.round(4), round(float(loss), 4))
```

**3. 양쪽을 다 보는 BERT, 앞만 보는 GPT**

의사코드

```
bert = 4 곱하기 4 짜리 전부 1 인 표
gpt = 같은 크기에서 아래쪽 삼각형만 1 인 표
각각 다 더해서 볼 수 있는 칸 수를 센다
gpt 는 줄마다 합을 따로 찍어 본다
```

```python
n = 4

bert_mask = np.ones((n, n))
gpt_mask = np.tril(np.ones((n, n)))

bert_seen = int(bert_mask.sum())
gpt_seen = int(gpt_mask.sum())

print(gpt_mask)
print(bert_seen, gpt_seen, gpt_mask.sum(axis=1))
```

**4. BERT 를 숫자로 읽기**

의사코드

```
base 헤드 차원 = 768 나누기 12
large 헤드 차원 = 1024 나누기 16
파라미터 배수 = 340 나누기 110
세 값을 찍어서 비교한다
```

```python
base = {"layers": 12, "hidden": 768, "heads": 12, "params_m": 110}
large = {"layers": 24, "hidden": 1024, "heads": 16, "params_m": 340}

base_head_dim = base["hidden"] // base["heads"]
large_head_dim = large["hidden"] // large["heads"]
param_ratio = large["params_m"] / base["params_m"]

print(base_head_dim, large_head_dim, round(param_ratio, 4))
```

**5. 몸통은 그대로, 과제 헤드만 갈아 끼우기**

의사코드

```
인코더 출력 자리에 (4, 16, 768) 짜리 0 텐서를 둔다
분류 헤드 = Linear(768, 2), [CLS] 자리 한 줄에만 씌운다
태깅 헤드 = Linear(768, 9), 모든 자리에 씌운다
파라미터 수 = 768 곱하기 클래스 수 더하기 클래스 수
몸통 대비 몇 퍼센트인지 나눠 본다
```

```python
enc_out = torch.zeros(4, 16, 768)    # (문장 4개, 토큰 16개, 768칸)

cls_head = nn.Linear(768, 2)         # 문장 하나 분류
tag_head = nn.Linear(768, 9)         # 토큰마다 꼬리표 9종

cls_logits = cls_head(enc_out[:, 0])
tag_logits = tag_head(enc_out)

cls_params = 768 * 2 + 2
body_params = 110_000_000
head_share = 100 * cls_params / body_params

print(tuple(cls_logits.shape), tuple(tag_logits.shape))
print(cls_params, round(head_share, 4))
```

**6. 미세조정 레시피를 걸음 수로 바꾸기**

의사코드

```
한 에폭 걸음 수 = 데이터 수 나누기 배치 크기, 올림
전체 걸음 수 = 한 에폭 걸음 수 곱하기 에폭 수
워밍업 걸음 수 = 전체 걸음 수 곱하기 0.1, 버림
사전 학습 학습률 = 미세조정 학습률 곱하기 100
```

```python
n_train, batch, epochs = 15000, 32, 2
lr_finetune = 2e-5

steps_per_epoch = math.ceil(n_train / batch)
total_steps = steps_per_epoch * epochs
warmup_steps = int(0.1 * total_steps)
lr_pretrain = lr_finetune * 100

print(steps_per_epoch, total_steps, warmup_steps, lr_pretrain)
```

**7. 1 퍼센트만 움직이기**

의사코드

```
움직이는 수 = 몸통 수 곱하기 0.01
몇 배 적게 움직이나 = 몸통 수 나누기 움직이는 수
전체 미세조정 저장량 = 몸통 수 곱하기 과제 수
PEFT 저장량 = 몸통 수 더하기 움직이는 수 곱하기 과제 수
두 저장량을 나눈다
```

```python
body = 110_000_000
tasks = 10

trainable = int(0.01 * body)
fewer_times = body // trainable
full_store = body * tasks
peft_store = body + trainable * tasks

print(trainable, fewer_times)
print(full_store, peft_store, round(full_store / peft_store, 4))
```

**8. 혼동 행렬에서 네 가지 숫자 구하기**

의사코드

```
맞힌 부정 = 0.874 곱하기 2516, 반올림
맞힌 긍정 = 0.874 곱하기 2484, 반올림
정확도 = (둘의 합) 나누기 5000
작은 표에서 정확도 = 대각선 합 나누기 전체
정밀도 = 오른쪽 아래 나누기 오른쪽 열 합
재현율 = 오른쪽 아래 나누기 아랫줄 합
F1 = 2 곱하기 정밀도 곱하기 재현율 나누기 (정밀도 더하기 재현율)
```

```python
rec_neg, n_neg = 0.874, 2516
rec_pos, n_pos = 0.874, 2484

correct_neg = round(rec_neg * n_neg)
correct_pos = round(rec_pos * n_pos)
accuracy = (correct_neg + correct_pos) / 5000

cm = np.array([[45, 5],
               [10, 40]])   # 줄: 실제 부정, 실제 긍정 / 칸: 부정이라 부름, 긍정이라 부름

acc2 = (cm[0, 0] + cm[1, 1]) / cm.sum()
precision = cm[1, 1] / cm[:, 1].sum()
recall = cm[1, 1] / cm[1].sum()
f1 = 2 * precision * recall / (precision + recall)

print(correct_neg, correct_pos, accuracy)
print(acc2, round(precision, 4), recall, round(f1, 4))
```

**9. 레이블을 늘리면 얼마나 좋아질까**

의사코드

```
상승폭 = (뒤 정확도 빼기 앞 정확도) 곱하기 100
데이터 배수 = 뒤 레이블 수 나누기 앞 레이블 수
성능 비율 = 500 정확도 나누기 15000 정확도, 곱하기 100
데이터 비율 = 500 나누기 15000, 곱하기 100
```

```python
acc = {500: 0.8050, 2000: 0.8418, 15000: 0.8740}

gain1 = 100 * (acc[2000] - acc[500])
gain2 = 100 * (acc[15000] - acc[2000])
cost1 = 2000 / 500
cost2 = 15000 / 2000

share_perf = 100 * acc[500] / acc[15000]
share_data = 100 * 500 / 15000

print(round(gain1, 2), round(gain2, 2), cost1, cost2)
print(round(share_perf, 1), round(share_data, 1))
```

**10. T5 의 스팬 손상 만들어 보기**

의사코드

```
i 를 0 부터 시작해 토큰을 훑는다
i 가 다음 스팬의 시작이면: 인코더에 센티널을 넣고, 디코더에 센티널과 지운 토큰들을 넣고, i 를 스팬 끝으로 옮긴다
아니면: 인코더에 토큰을 그대로 넣고 i 를 하나 늘린다
다 끝나면 디코더 끝에 센티널 하나를 더 붙인다
```

```python
tokens = ["Thank", "you", "for", "inviting", "me", "to", "your", "party", "last", "week"]
spans = [(2, 4), (8, 9)]          # 반열린 구간. (2, 4) 는 for, inviting
sentinels = ["<X>", "<Y>", "<Z>"]

enc, dec, i, k = [], [], 0, 0
while i < len(tokens):
    if k < len(spans) and i == spans[k][0]:
        enc.append(sentinels[k])
        dec.append(sentinels[k])
        dec += tokens[spans[k][0]:spans[k][1]]
        i = spans[k][1]
        k += 1
    else:
        enc.append(tokens[i])
        i += 1
dec.append(sentinels[k])

print("encoder input :", " ".join(enc))
print("decoder target:", " ".join(dec))
```